This example shows how to generate Feff Path files for EXAFS Paths

First, we will use the CIF library from larixite (installed with Larch, but also available independently) to look up and extract CIF files from the American Mineralogical Crystal Structure Database (AMCSD) and use that to generate the input needed for a Feff calculation.

Second, we will run Feff to generate the "Path files", representing the EXAFS contribution for each photo-electron scattering path.

Finally, we will add some of these paths together and compare them to real EXAFS data.

This example uses Bokeh for plotting, which produces interactive plots in the Jupyter notebook.

All of these steps can also be done in the Larix GUI, and we'll follow some of the conventions used there so that this can be used as a guide for going back and forth between using Larix and doing similar work programmatically.

This example will use the Zn K-edge of ZnSe data.

In [3]:
# step 0: some basic imports
import os
import numpy as np
from larch.io import read_ascii
from larch.xafs import pre_edge, autobk, xftf, feffrunner, feffpath, ff2chi
# import plotting functions for Bokenh
from larch.plot.bokeh_xafsplots import (plot_mu, plot_chik, plot_chir, plot_chifit, multi_plot, plotlabels, set_label_weight)

# import tools to work with the AMCSD and convert a CIF structure to Feff.inp
from larixite import get_amcsd, cif2feffinp

In [4]:
# step 1: create a connection to the CIF database
cifdb = get_amcsd()

In [5]:
# step 2: look up CIF structure using `cifdb.find_cifs`
# this supports several arguments:
#
# find_cifs(
#     id=None,
#     mineral_name=None,
#     author_name=None,
#     journal_name=None,# 
#     contains_elements=None,    <-  elements to include
#     excludes_elements=None,    
#     strict_contains=False,     <-  whether to *only* include 'contains_elemnents'
#     full_occupancy=False,      <-  only elements with full occupancy
#     max_matches=1000)


# to work with other Notebook examples, we'll look for ZnSe structures: 
# fortunately, there are only two of these, but unfortunately, they are two different structures:

structs = cifdb.find_cifs(contains_elements=['Zn', 'Se'], strict_contains=True)
for struct in structs:
    print(struct.ams_id, struct.formula, struct.hm_symbol, struct.publication)


11535 Zn Se F -4 3 m CifPublication(id=907, journalname='Crystal Structures', year=1963, volume='1', page_first='85', page_last='237', authors=('Wyckoff R W G',))
11557 Zn Se P 63 m c CifPublication(id=907, journalname='Crystal Structures', year=1963, volume='1', page_first='85', page_last='237', authors=('Wyckoff R W G',))


We would like to calculate the EXAFS paths for both structures, and compare them.  
Here, we'll focus on the Zn K edge.

First, we can generate the Feff.inp file for each structure using cif2feffinp.   This also supports several arguments:

cif2feffinp(
    ciftext,
    absorber,
    template=None,
    edge=None,
    cluster_size=8.0,
    absorber_site=None,
    extra_titles=None,
    with_h=False,
    version8=True,
    rng_seed=None,
    cifid=None
)
    convert CIF text to Feff8 or Feff6l input file

    Arguments
    ---------
      ciftext (string):         text of CIF file or name of the CIF file.
      absorber (string or int): atomic symbol or atomic number of absorbing element
                                (see Note 1)
      edge (string or None):    edge for calculation (see Note 2)     [None]
      cluster_size (float):     size of cluster, in Angstroms         [8.0]
      absorber_site (None or int):  index of site for absorber (see Note 3) [None]
      extra_titles (list of str or None): extra title lines to include [None]
      with_h (bool):            whether to include H atoms [False]
      version8 (bool):          whether to write Feff8l input (see Note 5)[True]
      rng_seed (int or None):   seed for RNG to get reproducible occupancy selections [None]
    Returns
    -------
      text of Feff input file

    Notes
    -----
      1. absorber is the atomic symbol or number of the absorbing element, and
         must be an element in the CIF structure. If absorber_site is None (default),
         the first site for that element will be used, as found from cluster.atom_sites.

      2. If edge is a string, it must be one of 'K', 'L', 'M', or 'N' edges (note
         Feff6 supports only 'K', 'L3', 'L2', and 'L1' edges). If edge is None,
         it will be assigned to be 'K' for absorbers with Z < 58 (Ce, with an
         edge energy < 40 keV), and 'L3' for absorbers with Z >= 58.
      3. for structures with multiple sites for the absorbing atom, the site
         can be selected by the order in which they are listed in the sites
         list. This depends on the details of the CIF structure, which can be
         found with `cif_sites(ciftext)`, starting counting by 1.
      5. if version8 is False, outputs will be written for Feff6l



In [6]:
# step 3: get the text of these two Feff.inp texts:

feffinp_znse_cubic = cif2feffinp(structs[0].ciftext, absorber='Zn', edge='K', cluster_size=7.0)
feffinp_znse_hexag = cif2feffinp(structs[1].ciftext, absorber='Zn', edge='K', cluster_size=7.0)

In [7]:
# step 4: save these and run feff

# In principle, the feff files can be run anyway.  
# But Feff writes a lot of files with pre-determined names, so it is best to have each run in its own folder.  
# And: Larix has a place where it will look for Feff files by default, so we'll use that convention so that
# Larix might also be able to use these results

from pathlib import Path
from larch import site_config

feff_folder = Path(site_config.user_larchdir, 'feff')

for foldername, feffinp in (('ZnSe_cubic', feffinp_znse_cubic),
                            ('ZnSe_hexag', feffinp_znse_hexag)):
    folder = Path(feff_folder, foldername)
    if not folder.exists():
        folder.mkdir()  # create this folder
    with open(Path(folder, 'feff.inp'), 'w') as fh:
        fh.write(feffinp)

# now run feff for each folder - this can take a little bit of time
for foldername in ('ZnSe_cubic', 'ZnSe_hexag'): 
    print(f"run Feff: {foldername}")
    feffrunner(feffinp=Path(feff_folder, foldername, 'feff.inp'), verbose=False).run(exe='feff8l')
print("done")

# note: running feff may leave us in that runtime folder, so we'll move back to where we started
ipython_instance = get_ipython()
os.chdir(ipython_instance.starting_dir)


run Feff: ZnSe_cubic
run Feff: ZnSe_hexag
done


In [8]:
# step 5: now read in a few of the Feff data files and generate EXAFS spectra

# a function to get paths up to some distance 
def get_path_list(folder, reff_max=5.0):
    pathlist = []
    for i in range(1, 200):
        fpath = Path(folder, f'feff{i:04}.dat')
        if fpath.exists():
            tpath = feffpath(fpath, s02=0.9, e0=0.0, sigma2=0.002)
            if tpath.reff < reff_max:
                pathlist.append(tpath)
    return pathlist

znse_cubic_paths = get_path_list(Path(feff_folder, 'ZnSe_cubic'), reff_max=5.0)

znse_hexag_paths = get_path_list(Path(feff_folder, 'ZnSe_hexag'), reff_max=5.0)

print("Paths ", len(znse_cubic_paths), len(znse_hexag_paths))

znse_cubic  = ff2chi(znse_cubic_paths)
xftf(znse_cubic, kmin=2, kmax=13, dk=4, kweight=2, window='hanning')

znse_hexag  = ff2chi(znse_hexag_paths)
xftf(znse_hexag, kmin=2, kmax=13, dk=4, kweight=2, window='hanning')

multi_plot([{'xdata':znse_hexag.k, 'ydata':znse_hexag.chi*znse_hexag.k**2, 'label': 'hex ZnSe'},
            {'xdata':znse_cubic.k, 'ydata':znse_cubic.chi*znse_cubic.k**2, 
             'label': 'cubic ZnSe','xlabel': plotlabels.k, 
             'ylabel': set_label_weight(plotlabels.chikw, 2) }])


multi_plot([{'xdata':znse_hexag.r, 'ydata':znse_hexag.chir_mag, 'label': 'hex ZnSe'},
            {'xdata':znse_cubic.r, 'ydata':znse_cubic.chir_mag, 
             'label': 'cubic ZnSe','xlabel': plotlabels.r, 
             'ylabel': set_label_weight(plotlabels.chir, 2) }])


Paths  7 14
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


Loading BokehJS ...

Loading BokehJS ...

In [9]:
# The local structures around Zn for those two crystal structures are pretty similar. 
# The cubic phase has
#     4 Zn-Se  2.45 Ang
#    12 Zb-Zn  4.01 Ang
# while the hexagonal phase has
#     3 Zn-Se  2.42 Ang
#     1 Zn-Se  2.51 Ang
#     6 Zn-Zn  3.98 Ang
#     6 Zn-Zn  3.99 Ang
#  Given the resolution limits in EXAFS data, those might be hard to distinguish.


#  We can compare to some ZnSe data
znse_data  = read_ascii('../xafsdata/znse_zn_xafs.001', labels=['energy', 'time', 'i0', 'itrans'])
znse_data.mu = -np.log(znse_data.itrans/znse_data.i0)
pre_edge(znse_data)
autobk(znse_data, rbkg=1.0, kw=2)

xftf(znse_data, kmin=2, kmax=13, dk=4, kweight=2, window='hanning')

plot_chik(znse_data, kweight=2)

Loading BokehJS ...

In [10]:

multi_plot([{'xdata':znse_data.r, 'ydata':znse_data.chir_mag, 'label': 'data'},
            {'xdata':znse_cubic.r, 'ydata':znse_cubic.chir_mag,
             'label': 'cubic ZnSe','xlabel': plotlabels.r},
             {'xdata':znse_hexag.r, 'ydata':znse_hexag.chir_mag,
             'label': 'hex ZnSe','xlabel': plotlabels.r}])
           


Loading BokehJS ...

In [ ]:
# at this point, either structure seems plausible. 

# In another notebook, we'll set up to model these EXAFS for these two cases.


In [19]:
# but to set that up, we'll first write the ZnSe chi(k) data directly to a data file
# The processing steps above created a "journal" in the `znse_data` group, and we'll 
# include that as part of the header
from larch.io import write_ascii

header = ['Data from znse_zn_xafs.001', 'original header:']
header.extend(znse_data.header)
header.append('processing steps')
header.extend([f'{key}: {val}' for key, val in znse_data.journal.items()])

write_ascii('znse_chi.dat', znse_data.k, znse_data.chi, label='k chi', header=header)

wrote to file 'znse_chi.dat'
